# Embeddings

Embeddings are the foundation of every retrieval-augmented generation system.
In this notebook you will learn what embeddings are, how to generate them with
SentenceTransformer, and how to use cosine similarity to measure semantic
closeness between product names, category labels, and policy terms from the
ShopSmart India knowledge base. Understanding this layer is critical: every
retrieval decision in CatalogueIQ — whether the system finds the right product
spec or the right policy clause — ultimately depends on the quality and
behaviour of these dense vector representations.

### Concepts Covered

- What an embedding is and why dense vectors capture semantic meaning better than keyword overlap
- How to load `SentenceTransformer('all-MiniLM-L6-v2')` and embed ShopSmart India domain terms
- Cosine similarity: why it is preferred over Euclidean distance for text embeddings
- Why "earphones" and "earbuds" are close in embedding space but "earbuds" and "kurta" are not
- Visualising a 2D PCA projection of product names, category names, and policy terms together
- Comparing semantically similar vs. dissimilar ShopSmart India term pairs as a sanity check
- How embedding model choice affects retrieval quality for structured product data vs. unstructured policy prose

## Model Used: `all-MiniLM-L6-v2`

The model being used is **all-MiniLM-L6-v2**, loaded via the `SentenceTransformer` class from the `sentence-transformers` library.

### Key Details

- **Type**: A MiniLM-based sentence embedding model, distilled from a larger transformer to be small and fast while retaining good semantic quality.
- **Embedding dimension**: `384` (the `model.get_sentence_embedding_dimension()` call in the code will print this).
- **Local/offline**: Runs entirely on your machine — no API calls to OpenAI, Anthropic, etc. This aligns with the code comment `# no API calls`, suggesting it's meant to be a free, local baseline for generating embeddings before comparing against API-based embedding models later in the notebook series.
- **Use case**: Popular for tasks like semantic search, clustering, and similarity comparison — which fits with the `cosine_similarity` and `PCA` imports (likely for visualizing embeddings in 2D/3D and measuring similarity between sentences).
- **Source**: From the `sentence-transformers` project (built on Hugging Face `transformers`). On first run, it downloads the model weights (~80MB) from the Hugging Face Hub and caches them locally.

### Why It's a Good Fit for NB-01

`all-MiniLM-L6-v2` is a strong teaching choice — it lets learners experiment with embeddings without needing API keys or incurring costs, before you introduce API-based embedding models later in the curriculum.

In [ ]:
# ── SETUP ──────────────────────────────────────────────────────
# No API key needed in NB-01: all computation is local.

# import os
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.patches as mpatches
# from sklearn.decomposition import PCA
# from sklearn.metrics.pairwise import cosine_similarity
# from sentence_transformers import SentenceTransformer

# # Model used throughout NB-01 (no API calls)
# EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# print(f"Loading embedding model: {EMBEDDING_MODEL}")
# model = SentenceTransformer(EMBEDDING_MODEL)
# print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())


In [1]:
import os
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ============================================================
# Local embedding model (sentence-transformers)
# ============================================================
LOCAL_EMBEDDING_MODEL = "all-MiniLM-L6-v2"

print(f"Loading local embedding model: {LOCAL_EMBEDDING_MODEL}")
model = SentenceTransformer(LOCAL_EMBEDDING_MODEL)
embedding_dim = model.get_sentence_embedding_dimension()
print("Model loaded. Embedding dimension:", embedding_dim)

def get_embeddings(texts):
    """Returns a numpy array of embeddings for a list of texts using sentence-transformers."""
    if isinstance(texts, str):
        texts = [texts]
    return model.encode(texts)

/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI/week2-mac/.venv_rag/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading local embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7702.69it/s]


Model loaded. Embedding dimension: 384


/var/folders/_g/x5x8p4910pzdcn9kd8gl4lrm0000gn/T/ipykernel_17627/792616618.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = model.get_sentence_embedding_dimension()


### Step 1 — Embedding ShopSmart India Domain Terms

We will embed three groups of terms that represent the three layers of the
CatalogueIQ knowledge base:

1. **Product terms** — names and colloquial synonyms used by shoppers
2. **Category labels** — the taxonomy nodes in the ShopSmart India category tree
3. **Policy terms** — key phrases from the returns policy and seller guide

After embedding all three groups we can see whether the model has learnt that
"earphones" and "earbuds" are near-synonyms, and whether product terms cluster
away from policy terms in vector space.

In [2]:
# ── DOMAIN TERMS FROM SHOPSMRT INDIA ─────────────────────────

# Group 1: product names and shopper synonyms
product_terms = [
    "boAt Airdopes 141 earbuds",
    "Sony WF-C500 earbuds",
    "Jabra ANC earbuds",
    "wireless earphones",           # colloquial synonym
    "TWS earbuds",                  # category abbreviation
    "in-ear headphones",            # another synonym
    "Samsung Galaxy Smartwatch",
    "noise cancelling headphones",
    "Libas cotton kurta",
    "Biba Anarkali suit",
    "Philips air fryer",
    "Prestige mixer grinder",
]

# Group 2: ShopSmart India category labels
category_terms = [
    "Electronics",
    "Fashion",
    "Home and Kitchen",
    "Beauty",
    "Books",
    "Earbuds and Headphones",
    "Kurtas and Kurtis",
    "Air Fryers",
    "Smartwatches",
]

# Group 3: policy and seller guide terms
policy_terms = [
    "return window 7 days",
    "30 day return policy",
    "refund timeline",
    "ShopSmart A-to-Z Guarantee",
    "seller verified",
    "image resolution requirements",
    "GST registration",
    "Big Billion Days sale",
    "final sale no returns",
    "warranty claim",
]

all_terms = product_terms + category_terms + policy_terms
labels = (
    ["product"] * len(product_terms) +
    ["category"] * len(category_terms) +
    ["policy"] * len(policy_terms)
)

print(f"Total terms to embed: {len(all_terms)}")

# Generate embeddings — returns shape (N, 384) for all-MiniLM-L6-v2
embeddings = model.encode(all_terms, show_progress_bar=True)
print(f"Embedding matrix shape: {embeddings.shape}")


Total terms to embed: 31


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.41it/s]

Embedding matrix shape: (31, 384)


In [3]:
# ── DISPLAY ONE EMBEDDING VALUE ──────────────────────────────

# Show the full embedding vector for the first term
print(f"\nTerm: '{all_terms[0]}'")
print(f"Label: {labels[0]}")
print(f"Embedding vector (384 dims):\n{embeddings[0]}")

# Just the first few dimensions, for a quick peek
print(f"\nFirst 10 values: {embeddings[0][:10]}")

# A single scalar value from that vector (e.g., dimension 0)
print(f"\nSingle value — embeddings[0][0]: {embeddings[0][0]}")


Term: 'boAt Airdopes 141 earbuds'
Label: product
Embedding vector (384 dims):
[-3.69760394e-02  1.85434446e-02  3.53143131e-03 -1.34988185e-02
 -3.70238759e-02 -7.74916261e-02  5.95261827e-02  2.35074032e-02
 -1.26353428e-01  2.14693043e-03  4.72359173e-02 -7.82370046e-02
 -1.13589503e-02  8.67439061e-03 -4.45370264e-02  8.70122612e-02
  1.16957435e-02 -4.77400534e-02  6.97562820e-04 -5.03108930e-03
 -5.09332828e-02  1.23601437e-01  3.09766885e-02 -2.13817004e-02
 -7.34671727e-02  3.64338569e-02 -2.47838013e-02 -1.84325688e-02
 -5.31779565e-02 -6.22104555e-02 -8.20727553e-03  1.61549225e-02
  2.94988062e-02 -3.73995155e-02  3.65963280e-02 -1.88892931e-02
  7.47754145e-03 -4.77676950e-02 -1.87549796e-02  5.76434918e-02
 -6.81604892e-02  3.01790461e-02 -2.17413176e-02  6.30961126e-03
 -3.45630385e-02 -1.46961194e-02 -9.44780260e-02 -1.27401287e-02
  9.14171040e-02  1.18977278e-01  6.82477504e-02 -3.99155915e-02
  2.93350182e-02  3.47527303e-02  5.33743808e-03 -3.91390994e-02
 -1.2994551

### Step 2 — Cosine Similarity Between Domain Term Pairs

Before visualising the full embedding space we check specific term pairs that
matter for CatalogueIQ's retrieval quality. A shopper query like "earphones
under 2000" should retrieve boAt Airdopes 141 earbuds — this only works if
the embedding model places "earphones" close to "earbuds" in vector space.

In [ ]:
# ── PAIRWISE SIMILARITY FOR KEY SHOPMART INDIA TERM PAIRS ────

def cosine_sim(a, b):
    """Cosine similarity between two 1-D embedding vectors."""
    # np.dot / (||a|| * ||b||) — the standard formula
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def embed_and_compare(term_a, term_b):
    vec_a = model.encode(term_a)
    vec_b = model.encode(term_b)
    # print(f"\nEmbedding-1 for '{term_a}':\n{vec_a}")
    #print(f"\nEmbedding-2 for '{term_b}':\n{vec_b}")
    sim = cosine_sim(vec_a, vec_b)
    print(f"  '{term_a}'  ↔  '{term_b}'  →  similarity = {sim:.4f}")
    return sim

print("\n─── Semantically SIMILAR pairs (expect high similarity) ───")
# embed_and_compare("earphones",          "earbuds")
# embed_and_compare("earphones",          "TWS earbuds")
# embed_and_compare("return policy",      "refund policy")
# embed_and_compare("kurtis",             "kurta")
# embed_and_compare("mobile phone",       "smartphone")
embed_and_compare("noise cancelling",   "ANC")

# print("\n─── Semantically DISSIMILAR pairs (expect low similarity) ───")
# embed_and_compare("earbuds",            "kurta")
# embed_and_compare("battery life",       "refund timeline")
# embed_and_compare("mixer grinder",      "GST registration")
# embed_and_compare("air fryer capacity", "seller verified")

print("\n💡 Observations:")
print("   High-similarity pairs support ShopSmart India's synonym expansion strategy.")
print("   Low-similarity pairs confirm the model separates product facts from policy text.")


### Step 3 — 2D PCA Visualisation of the Embedding Space

This code takes the 384-dimensional embeddings from your ShopSmart domain terms and visualizes them on a 2D scatter plot, colored by category, so you can see whether semantically related terms cluster together.

**What PCA is**: PCA (Principal Component Analysis) is a dimensionality-reduction technique. Your embeddings live in 384-dimensional space — impossible to plot or eyeball directly. PCA finds the 2 (or 3) directions in that high-dimensional space along which the data varies the most, and projects every point onto just those directions. Think of it like photographing a 3D object from the angle that shows its most distinctive silhouette — you lose some detail, but you keep the most informative view.

Reducing 384 dimensions to 2 using PCA lets us visually inspect whether the three document layers — products, categories, and policy terms — form distinct clusters. In CatalogueIQ this matters because we want a product query to land near product chunks, not near policy sections.

In [ ]:
# ── PCA PROJECTION ────────────────────────────────────────────


import os
os.makedirs("output", exist_ok=True)

# Reduce to 2 dimensions for visualisation
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(embeddings)       # shape: (N, 2)

print(f"Explained variance by PC1 + PC2: "
      f"{pca.explained_variance_ratio_.sum()*100:.1f}%")

# Colour scheme for the three groups
colour_map = {"product": "#1f77b4", "category": "#ff7f0e", "policy": "#2ca02c"}
colours = [colour_map[l] for l in labels]

fig, ax = plt.subplots(figsize=(13, 9))

scatter = ax.scatter(
    coords_2d[:, 0], coords_2d[:, 1],
    c=colours, s=80, alpha=0.85, edgecolors="white", linewidths=0.5
)

# Annotate every point with its term — offset slightly to avoid overlap
for i, term in enumerate(all_terms):
    # Shorten long product names for readability
    label_text = term if len(term) < 28 else term[:26] + "…"
    ax.annotate(
        label_text,
        (coords_2d[i, 0], coords_2d[i, 1]),
        textcoords="offset points", xytext=(6, 4),
        fontsize=7, alpha=0.9
    )

# Legend
patches = [mpatches.Patch(color=v, label=k.capitalize())
           for k, v in colour_map.items()]
ax.legend(handles=patches, loc="lower right", fontsize=10)
ax.set_title(
    "PCA Projection of ShopSmart India Domain Embeddings\n"
    "(all-MiniLM-L6-v2 → 384 dims → 2D PCA)",
    fontsize=13
)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig("output/nb01_pca_embeddings.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to output/nb01_pca_embeddings.png")


### Step 4 — Similarity Matrix for Product Synonym Groups

A similarity heatmap across the synonym group (earphones, earbuds, TWS, in-ear
headphones, wireless earphones) tells us directly which terms the model treats
as interchangeable. This drives the synonym expansion dictionary we will build
in NB-06.

In [ ]:
# ── SIMILARITY HEATMAP FOR EARBUD SYNONYMS ───────────────────

synonym_group = [
    "earphones",
    "earbuds",
    "TWS earbuds",
    "in-ear headphones",
    "wireless earphones",
    "ANC earbuds",
    "noise cancelling earphones",
]

syn_embeddings = model.encode(synonym_group)

# Compute full pairwise cosine similarity matrix
sim_matrix = cosine_similarity(syn_embeddings)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap="YlGn", vmin=0.5, vmax=1.0)
plt.colorbar(im, ax=ax, label="Cosine Similarity")

ax.set_xticks(range(len(synonym_group)))
ax.set_yticks(range(len(synonym_group)))
ax.set_xticklabels(synonym_group, rotation=35, ha="right", fontsize=9)
ax.set_yticklabels(synonym_group, fontsize=9)

# Annotate each cell with the score
for i in range(len(synonym_group)):
    for j in range(len(synonym_group)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}",
                ha="center", va="center", fontsize=8,
                color="black" if sim_matrix[i, j] < 0.85 else "white")

ax.set_title("Earbud Synonym Similarity Matrix\n(used to validate ShopSmart India synonym expansion)", fontsize=11)
plt.tight_layout()
plt.savefig("output/nb01_synonym_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Heatmap saved to output/nb01_synonym_heatmap.png")


In [ ]:
# ── EXPERIMENT ────────────────────────────────────────────────
# Run each exercise, observe the output, and note your findings.

# EXERCISE 1 — Model comparison
# Swap EMBEDDING_MODEL to 'paraphrase-MiniLM-L3-v2' (smaller/faster)
# or 'all-mpnet-base-v2' (larger/slower).
# Re-run Step 2 similarity pairs. Do the similarity scores change?
# Does "earphones" ↔ "earbuds" go up or down?
# Which model would you choose for a 200,000-product catalogue and why?

# EXERCISE 2 — Hinglish / code-switched queries
# ShopSmart India shoppers often type in Hinglish.
# Add these terms to all_terms and re-run the PCA:
#   "sasta earphone" (cheap earphones in Hindi slang)
#   "accha smartwatch" (good smartwatch)
#   "wapsi kaise karein" (how do I return — transliterated Hindi)
# Where do they appear in the 2D plot relative to their English equivalents?
# What does this tell you about how the model handles Hinglish queries?

# EXERCISE 3 — Category boundary probing
# Embed these ambiguous ShopSmart India terms:
#   "fitness band" — is it closer to Smartwatches or Earbuds?
#   "laptop stand" — is it closer to Electronics or Home & Kitchen?
#   "protein powder" — is it closer to Beauty or Books?
# Print the cosine similarity to the two nearest category embeddings.
# How would you use this to build an automatic category-assignment tool?

print("Exercises loaded. Edit the cells above and re-run to explore.")


### Key Takeaway

Embeddings transform text into dense vectors where semantic similarity maps to
geometric closeness — "earphones" and "earbuds" land near each other because
the model has seen them used in similar contexts, even though they share no
characters. The PCA projection confirmed that product terms, category labels,
and policy text form loosely separated clusters, which is encouraging: it means
a product query is unlikely to accidentally retrieve a policy chunk, and vice
versa.

In **NB-02** you will define what "good" retrieval and generation looks like
*before* building anything, using RAGAS evaluation metrics and a golden test set
drawn from the five CatalogueIQ query types.